# 🤖 Reranker Model Evaluation for FinanceRAG

## 📋 Overview

Notebook này đánh giá **reranker models** tốt nhất cho toàn bộ dataset.

### 🎯 Objectives:
1. Test multiple reranker models
2. Compare NDCG@10 scores across datasets
3. Analyze model characteristics (speed, accuracy)
4. Recommend the best reranker for production

### 🔧 Setup:
- **Embedding Model**: E5-small (best from notebook 0)
- **Chunking**: Pre-chunked optimal corpus (from notebook 4)
- **Hybrid Alpha**: 0.6 (default, can be updated from notebook 1)
- **Sampling**: 30 queries per dataset (like notebook 0)
- **Rerankers to Test**:
  - BGE-reranker-base
  - BGE-reranker-large
  - BGE-reranker-v2-m3
  - MS-MARCO cross-encoders

---

## 📦 Setup and Imports

In [2]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
import time
import gc
import torch

warnings.filterwarnings('ignore')

# Embedding & Retrieval libraries
from sentence_transformers import SentenceTransformer, CrossEncoder
from FlagEmbedding import FlagReranker
from rank_bm25 import BM25Okapi
import faiss

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")
print(f"🖥️ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    print("   Running on CPU")
    device = 'cpu'

✅ Libraries imported successfully!
🖥️ CUDA available: True
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## ⚙️ Configuration

In [3]:
# Load configuration
from config import (
    DATA_DIR, OUTPUT_DIR, CHUNKED_CORPUS_DIR, CHUNKING_CONFIG_FILE,
    DATASETS, RERANKER_MODELS, EMBEDDING_MODEL,
    SAMPLE_QUERIES_PER_DATASET, MAX_CORPUS_SAMPLE_SIZE,
    TOP_K_RETRIEVAL, TOP_K_RERANK, TOP_K_FINAL,
    EMBED_BATCH_SIZE, RERANK_BATCH_SIZE,
    print_config, print_reranker_config
)

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

# Print configuration
print_config()
print_reranker_config()


📊 Configuration:
   Datasets: 7
   Alpha values to test: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
   Reranker models: 5

📂 Data Source:
   Using pre-chunked corpus: True
   Chunked corpus dir: ..\..\data\chunked_corpus
   Chunking config: ..\..\data\chunked_corpus\best_chunking_config_per_dataset.json

🎯 Retrieval Settings:
   Embedding model: intfloat/e5-small-v2
   Top-k retrieval: 100
   Top-k rerank: 50
   Top-k final: 10

⚡ Speed optimizations:
   Queries per dataset: 30
   Max corpus sample: 500
   Smart sampling: True

🤖 RERANKER EVALUATION CONFIG:
   Models to test:
      [1] BGE-reranker-base: Base BGE reranker (278M params)
      [2] BGE-reranker-large: Large BGE reranker (560M params)
      [3] BGE-reranker-v2-m3: Multilingual BGE v2 (568M params)
      [4] MiniLM-L6-cross: Fast cross-encoder (22M params)
      [5] MiniLM-L12-cross: Better cross-encoder (33M params)
   Total tests per dataset: 5
   Total tests: 35


## 💾 Memory Management

Adjust these if you encounter memory issues:


In [4]:
# Check available memory and adjust settings AGGRESSIVELY
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 GPU Memory: {total_mem:.2f} GB")
    
    # CONSERVATIVE SETTINGS for stability
    if total_mem < 12.0:
        print("   ⚠️ Reducing settings for memory safety...")
        SAMPLE_QUERIES_PER_DATASET = 15  # Reduce from 30
        TOP_K_RETRIEVAL = 30             # Reduce from 100
        TOP_K_RERANK = 20                # Reduce from 50
        EMBED_BATCH_SIZE = 16            # Reduce batch size
        RERANK_BATCH_SIZE = 8            # Reduce rerank batch
        print(f"   Adjusted settings:")
        print(f"     - Queries: {SAMPLE_QUERIES_PER_DATASET}")
        print(f"     - Top-K retrieval: {TOP_K_RETRIEVAL}")
        print(f"     - Top-K rerank: {TOP_K_RERANK}")
        print(f"     - Embed batch: {EMBED_BATCH_SIZE}")
        print(f"     - Rerank batch: {RERANK_BATCH_SIZE}")
    else:
        print("   ✅ Sufficient GPU memory (but using conservative mode)")
else:
    print("💻 Running on CPU")
    print("   ℹ️ This will be slower but more stable")
    # Even more conservative on CPU
    SAMPLE_QUERIES_PER_DATASET = 10
    TOP_K_RETRIEVAL = 20
    TOP_K_RERANK = 15

# Clear any existing cached memory
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
gc.collect()
print("✅ Memory cleared and settings configured")


💾 GPU Memory: 4.29 GB
   ⚠️ Reducing settings for memory safety...
   Adjusted settings:
     - Queries: 15
     - Top-K retrieval: 30
     - Top-K rerank: 20
     - Embed batch: 16
     - Rerank batch: 8
✅ Memory cleared and settings configured


## 📚 Import Shared Utilities

## 🩺 Pre-Run Memory Diagnostics

Check if system has enough resources before starting evaluation:


In [5]:
import psutil

print("🩺 SYSTEM DIAGNOSTICS")
print("="*80)

# CPU
print(f"\n💻 CPU:")
print(f"   Logical cores: {psutil.cpu_count(logical=True)}")
print(f"   Physical cores: {psutil.cpu_count(logical=False)}")
print(f"   Usage: {psutil.cpu_percent(interval=1)}%")

# RAM
ram = psutil.virtual_memory()
print(f"\n🧠 RAM:")
print(f"   Total: {ram.total / 1e9:.2f} GB")
print(f"   Available: {ram.available / 1e9:.2f} GB")
print(f"   Used: {ram.used / 1e9:.2f} GB ({ram.percent}%)")

if ram.available / 1e9 < 4.0:
    print("   ⚠️ WARNING: Less than 4GB RAM available!")
    print("   ⚠️ Notebook may crash. Close other applications.")

# GPU
if torch.cuda.is_available():
    print(f"\n🎮 GPU:")
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"   Total memory: {props.total_memory / 1e9:.2f} GB")
    print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"   Free: {(props.total_memory - torch.cuda.memory_reserved()) / 1e9:.2f} GB")
    
    if props.total_memory / 1e9 < 6.0:
        print("   ⚠️ WARNING: Less than 6GB GPU memory!")
        print("   ⚠️ Expect slower processing or potential crashes.")
        print("   ⚠️ Recommendation: Use smaller batch sizes or reduce sample count.")
else:
    print(f"\n💻 Running on CPU (no GPU available)")

# Recommendations
print(f"\n📋 RECOMMENDATIONS:")
estimated_peak = 8.0  # GB
if torch.cuda.is_available():
    total_gpu = props.total_memory / 1e9
    if total_gpu >= estimated_peak:
        print("   ✅ System should handle evaluation comfortably")
    elif total_gpu >= 6.0:
        print("   ⚠️ System is at the edge - use conservative settings")
        print("   ⚠️ Close all other GPU applications")
    else:
        print("   ❌ High risk of memory crash")
        print("   ❌ Strongly recommend reducing SAMPLE_QUERIES_PER_DATASET to 10")
        print("   ❌ And reducing TOP_K_RETRIEVAL to 20")
else:
    if ram.available / 1e9 >= 8.0:
        print("   ✅ CPU mode should work (slower)")
    else:
        print("   ⚠️ Low RAM - may struggle on CPU")

print("="*80)


🩺 SYSTEM DIAGNOSTICS

💻 CPU:
   Logical cores: 16
   Physical cores: 8
   Usage: 17.7%

🧠 RAM:
   Total: 16.54 GB
   Available: 1.54 GB
   Used: 15.00 GB (90.7%)
   ⚠️ WARNING: Less than 4GB RAM available!
   ⚠️ Notebook may crash. Close other applications.

🎮 GPU:
   Device: NVIDIA GeForce RTX 3050 Laptop GPU
   Total memory: 4.29 GB
   Allocated: 0.00 GB
   Cached: 0.00 GB
   Free: 4.29 GB
   ⚠️ WARNING: Less than 6GB GPU memory!
   ⚠️ Expect slower processing or potential crashes.
   ⚠️ Recommendation: Use smaller batch sizes or reduce sample count.

📋 RECOMMENDATIONS:
   ❌ High risk of memory crash
   ❌ Strongly recommend reducing SAMPLE_QUERIES_PER_DATASET to 10
   ❌ And reducing TOP_K_RETRIEVAL to 20


In [6]:
# Import data loading functions
from utils import (
    load_jsonl,
    load_prechunked_corpus,
    sample_queries,
    smart_sample_corpus,
    compute_ndcg,
    normalize_scores,
    hybrid_search,
    aggregate_chunk_scores,
    QRELS_MAPPING
)

print("✅ Utility functions imported from utils.py")

✅ Utility functions imported from utils.py


## 🤖 Load Embedding Model (Fixed)

In [7]:
print("Loading embedding model...")
print(f"Model: {EMBEDDING_MODEL}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print("✅ Embedding model loaded!")

Loading embedding model...
Model: intfloat/e5-small-v2
✅ Embedding model loaded!


## 🎯 Reranker Evaluation Function

In [8]:
def load_reranker(model_name: str, model_type: str, device: str):
    """
    Load reranker model based on type.
    
    Args:
        model_name: HuggingFace model name
        model_type: 'flag' for FlagReranker, 'cross' for CrossEncoder
        device: 'cuda' or 'cpu'
        
    Returns:
        Loaded reranker model
    """
    if model_type == 'flag':
        return FlagReranker(model_name, use_fp16=(device=='cuda'))
    else:  # cross-encoder
        return CrossEncoder(model_name, device=device)


def evaluate_reranker_on_dataset(
    dataset_name: str,
    reranker,  # Pass loaded reranker instead of loading inside
    reranker_name: str,
    reranker_type: str,
    embed_model,
    device: str,
    hybrid_alpha: float = 0.6
) -> Dict:
    """
    Evaluate a specific reranker model on a dataset.
    
    Args:
        dataset_name: Name of the dataset
        reranker: Pre-loaded reranker model
        reranker_name: HuggingFace reranker model name (for logging)
        reranker_type: 'flag' or 'cross'
        embed_model: Embedding model
        device: 'cuda' or 'cpu'
        hybrid_alpha: Hybrid search alpha
        
    Returns:
        Dict with NDCG@10 and timing metrics
    """
    start_time = time.time()
    
    try:
        # Load pre-chunked corpus
        chunks, chunking_method = load_prechunked_corpus(
            dataset_name,
            CHUNKED_CORPUS_DIR,
            CHUNKING_CONFIG_FILE
        )
        
        if not chunks:
            return {'ndcg_10': 0.0, 'error': 'Failed to load chunks'}
        
        # Build chunk-to-doc mapping
        chunk_to_doc = {}
        for c in chunks:
            chunk_id = c.get('_id', c.get('chunk_id', ''))
            doc_id = c.get('original_id', c.get('doc_id', chunk_id))
            chunk_to_doc[chunk_id] = doc_id
        
        chunk_texts = [c.get('text', '') for c in chunks]
        chunk_ids = [c.get('_id', c.get('chunk_id', '')) for c in chunks]
        
        # Load queries and qrels
        queries_path = DATA_DIR / f"{dataset_name}_queries.jsonl" / "queries.jsonl"
        queries = load_jsonl(queries_path)
        
        qrels_path = DATA_DIR / QRELS_MAPPING[dataset_name]
        qrels_dict = {}
        if qrels_path.exists():
            qrels_df = pd.read_csv(qrels_path, sep='\t')
            query_col = 'query_id' if 'query_id' in qrels_df.columns else 'query-id'
            corpus_col = 'corpus_id' if 'corpus_id' in qrels_df.columns else 'corpus-id'
            score_col = 'score' if 'score' in qrels_df.columns else 'relevance'
            
            for _, row in qrels_df.iterrows():
                qid = str(row[query_col])
                cid = str(row[corpus_col])
                score = int(row[score_col])
                if qid not in qrels_dict:
                    qrels_dict[qid] = {}
                qrels_dict[qid][cid] = score
        
        # Sample queries
        queries_sample = sample_queries(queries, qrels_dict, SAMPLE_QUERIES_PER_DATASET)
        
        # Encode chunks
        chunk_embeddings = embed_model.encode(
            chunk_texts,
            batch_size=EMBED_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        
        # Build FAISS index
        index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
        index.add(chunk_embeddings.astype('float32'))
        
        # Build BM25 index
        tokenized = [t.lower().split() for t in chunk_texts]
        bm25 = BM25Okapi(tokenized)
        
        # Process queries
        query_texts = [q.get('text', '') for q in queries_sample]
        query_ids = [q.get('_id', '') for q in queries_sample]
        
        query_embeddings = embed_model.encode(
            query_texts,
            batch_size=EMBED_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        
        # Retrieve and rerank
        results_dict = {}
        rerank_times = []
        
        for i, query_id in enumerate(query_ids):
            query_emb = query_embeddings[i]
            query_text = query_texts[i]
            
            # Hybrid search
            scores, chunk_indices = hybrid_search(
                query_emb, query_text, index, bm25, chunk_texts,
                TOP_K_RETRIEVAL, hybrid_alpha
            )
            
            # Aggregate chunks to documents
            doc_scores = {}
            for idx, score in zip(chunk_indices, scores):
                doc_id = chunk_to_doc[chunk_ids[idx]]
                if doc_id not in doc_scores:
                    doc_scores[doc_id] = []
                doc_scores[doc_id].append(float(score))
            
            doc_agg = aggregate_chunk_scores(doc_scores, 'max')
            sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:TOP_K_RERANK]
            
            # Rerank top documents
            candidate_ids = [d[0] for d in sorted_docs]
            candidate_texts = []
            for doc_id in candidate_ids:
                # Find chunks belonging to this doc
                doc_chunks = [chunk_texts[j] for j, cid in enumerate(chunk_ids) if chunk_to_doc[cid] == doc_id]
                candidate_texts.append(' '.join(doc_chunks)[:2048])
            
            pairs = [[query_text, t] for t in candidate_texts]
            
            # Time reranking
            rerank_start = time.time()
            
            if reranker_type == 'flag':
                rerank_scores = reranker.compute_score(pairs)
            else:  # cross-encoder
                rerank_scores = reranker.predict(pairs)
            
            rerank_time = time.time() - rerank_start
            rerank_times.append(rerank_time)
            
            if not isinstance(rerank_scores, list):
                rerank_scores = [rerank_scores]
            
            scored = list(zip(candidate_ids, rerank_scores))
            scored.sort(key=lambda x: x[1], reverse=True)
            
            # Store top-k final
            results_dict[query_id] = [doc_id for doc_id, _ in scored[:TOP_K_FINAL]]
        
        # Compute NDCG@10
        ndcg_scores = []
        for qid, retrieved in results_dict.items():
            if qid in qrels_dict:
                ndcg = compute_ndcg(retrieved, qrels_dict[qid], k=10)
                ndcg_scores.append(ndcg)
        
        avg_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
        avg_rerank_time = np.mean(rerank_times) if rerank_times else 0.0
        total_time = time.time() - start_time
        
        # Clean up
        del chunk_embeddings, query_embeddings, index, bm25, chunks, tokenized
        if device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
        
        return {
            'dataset': dataset_name,
            'reranker': reranker_name,
            'ndcg_10': float(avg_ndcg),
            'num_queries': len(ndcg_scores),
            'avg_rerank_time': float(avg_rerank_time),
            'total_time': float(total_time),
            'chunking_method': chunking_method
        }
    
    except Exception as e:
        # Clean up on error
        if device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
        return {
            'dataset': dataset_name,
            'reranker': reranker_name,
            'ndcg_10': 0.0,
            'error': str(e)
        }


print("✅ Reranker evaluation function defined")


✅ Reranker evaluation function defined


## 🚀 Run Reranker Evaluation

Test all reranker models across all datasets:

### ⚠️ EMERGENCY MODE: Test Single Reranker

If kernel keeps crashing, uncomment this cell to test ONLY the first reranker model:


In [9]:
# EMERGENCY MODE: Test only the fastest reranker first
# Uncomment these lines if you keep getting crashes:

# RERANKER_MODELS = [RERANKER_MODELS[3]]  # Test only MiniLM-L6-cross (fastest/smallest)
# DATASETS = ['financebench']  # Test only smallest dataset first

# print("⚠️ EMERGENCY MODE ACTIVATED")
# print(f"Testing only: {[m[1] for m in RERANKER_MODELS]}")
# print(f"On datasets: {DATASETS}")


In [10]:
results = []

print("\n" + "="*80)
print("🚀 STARTING RERANKER MODEL EVALUATION")
print("="*80)

# Show memory info
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory:")
    print(f"   Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

total_tests = len(DATASETS) * len(RERANKER_MODELS)
print(f"\n📊 Total tests to run: {total_tests}")
print(f"   Datasets: {len(DATASETS)}")
print(f"   Reranker models: {len(RERANKER_MODELS)}")

# CONSERVATIVE MODE: Process ONE reranker-dataset pair at a time with FULL cleanup
print(f"\n⚠️ ULTRA-CONSERVATIVE MODE:")
print(f"   Processing one model-dataset pair at a time")
print(f"   Full memory cleanup between each test")

pbar = tqdm(total=total_tests, desc="Overall progress")

for model_name, display_name, desc in RERANKER_MODELS:
    print(f"\n{'='*80}")
    print(f"🤖 Reranker: {display_name.upper()}")
    print(f"{'='*80}")
    
    for dataset in DATASETS:
        try:
            print(f"  📊 {dataset}...", end=' ', flush=True)
            
            # Determine model type
            if 'bge-reranker' in model_name.lower():
                model_type = 'flag'
            else:
                model_type = 'cross'
            
            # Load reranker for THIS test only
            reranker = None
            try:
                reranker = load_reranker(model_name, model_type, device)
                
                # Run evaluation
                result = evaluate_reranker_on_dataset(
                    dataset,
                    reranker,
                    model_name,
                    model_type,
                    embed_model,
                    device
                )
                
                result['display_name'] = display_name
                result['description'] = desc
                results.append(result)
                
                print(f"✅ NDCG@10: {result['ndcg_10']:.4f} ({result['avg_rerank_time']:.3f}s)")
                
            except Exception as e:
                print(f"❌ Error: {e}")
                results.append({
                    'dataset': dataset,
                    'reranker': model_name,
                    'display_name': display_name,
                    'ndcg_10': 0.0,
                    'error': str(e)
                })
            
            finally:
                # ULTRA-AGGRESSIVE CLEANUP after EACH test
                if reranker is not None:
                    del reranker
                
                # Clear all cached variables
                if 'result' in locals():
                    del result
                
                # Force GPU cleanup
                if device == 'cuda':
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                
                # Force Python garbage collection
                gc.collect()
                
                # Small pause to let memory settle
                import time
                time.sleep(0.5)
            
            pbar.update(1)
            
            # Monitor memory after cleanup
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated() / 1e9
                if allocated > 6.0:
                    print(f"      ⚠️ GPU memory after cleanup: {allocated:.2f} GB")
                    # Extra cleanup if memory is high
                    torch.cuda.empty_cache()
                    gc.collect()
                    time.sleep(1.0)
            
        except Exception as e:
            print(f"  ❌ Fatal error on {dataset}: {e}")
            results.append({
                'dataset': dataset,
                'reranker': model_name,
                'display_name': display_name,
                'ndcg_10': 0.0,
                'error': f'Fatal: {e}'
            })
            pbar.update(1)
            
            # Emergency cleanup
            if device == 'cuda':
                torch.cuda.empty_cache()
            gc.collect()
            time.sleep(1.0)

pbar.close()

print("\n" + "="*80)
print("✅ RERANKER EVALUATION COMPLETED!")
print("="*80)

# Final memory cleanup
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
gc.collect()



🚀 STARTING RERANKER MODEL EVALUATION

💾 GPU Memory:
   Total: 4.29 GB
   Allocated: 0.13 GB
   Cached: 0.15 GB

📊 Total tests to run: 35
   Datasets: 7
   Reranker models: 5

⚠️ ULTRA-CONSERVATIVE MODE:
   Processing one model-dataset pair at a time
   Full memory cleanup between each test


Overall progress:   0%|          | 0/35 [00:00<?, ?it/s]


🤖 Reranker: BGE-RERANKER-BASE
  📊 convfinqa... 

: 

## 📊 Results Analysis

In [ ]:
# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save raw results
results_df.to_csv(OUTPUT_DIR / 'reranker_evaluation_results.csv', index=False)
print(f"✅ Results saved to: {OUTPUT_DIR / 'reranker_evaluation_results.csv'}")

# Display results
print("\n📋 All Results:")
print(results_df[['dataset', 'display_name', 'ndcg_10', 'avg_rerank_time']].to_string(index=False))

## 🏆 Find Best Reranker Overall

In [ ]:
# Compute average NDCG per reranker
reranker_performance = results_df.groupby('display_name').agg({
    'ndcg_10': 'mean',
    'avg_rerank_time': 'mean',
    'total_time': 'mean'
}).reset_index()

reranker_performance = reranker_performance.sort_values('ndcg_10', ascending=False)

print("\n" + "="*80)
print("🏆 RERANKER MODEL RANKING")
print("="*80)
print(f"\n{'Rank':<6} {'Model':<25} {'Avg NDCG@10':<14} {'Avg Rerank Time':<18}")
print("-"*80)

for i, row in enumerate(reranker_performance.itertuples(), 1):
    print(f"{i:<6} {row.display_name:<25} {row.ndcg_10:<14.4f} {row.avg_rerank_time:<18.3f}s")

print("-"*80)

# Best model
best_model = reranker_performance.iloc[0]
print(f"\n🥇 BEST RERANKER: {best_model['display_name']}")
print(f"   Average NDCG@10: {best_model['ndcg_10']:.4f}")
print(f"   Average Rerank Time: {best_model['avg_rerank_time']:.3f}s")

# Save ranking
reranker_performance.to_csv(OUTPUT_DIR / 'reranker_ranking.csv', index=False)
print(f"\n✅ Ranking saved to: {OUTPUT_DIR / 'reranker_ranking.csv'}")

## 📈 Visualization: Reranker Performance

In [ ]:
# Bar plot: NDCG@10 by reranker
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Average NDCG@10
ax1 = axes[0]
reranker_performance_sorted = reranker_performance.sort_values('ndcg_10', ascending=True)
ax1.barh(reranker_performance_sorted['display_name'], reranker_performance_sorted['ndcg_10'])
ax1.set_xlabel('Average NDCG@10', fontsize=12)
ax1.set_title('Reranker Performance Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Speed vs Accuracy
ax2 = axes[1]
ax2.scatter(reranker_performance['avg_rerank_time'], reranker_performance['ndcg_10'], s=200, alpha=0.6)
for _, row in reranker_performance.iterrows():
    ax2.annotate(row['display_name'], 
                (row['avg_rerank_time'], row['ndcg_10']),
                fontsize=9, ha='right', va='bottom')
ax2.set_xlabel('Average Rerank Time (s)', fontsize=12)
ax2.set_ylabel('Average NDCG@10', fontsize=12)
ax2.set_title('Speed vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reranker_performance_comparison.png', dpi=300, bbox_inches='tight')
print(f"✅ Plot saved to: {OUTPUT_DIR / 'reranker_performance_comparison.png'}")
plt.show()

## 📊 Heatmap: Dataset vs Reranker

In [ ]:
# Create pivot table for heatmap
pivot_df = results_df.pivot(index='dataset', columns='display_name', values='ndcg_10')

# Plot heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='RdYlGn', 
            linewidths=0.5, cbar_kws={'label': 'NDCG@10'})
plt.title('Reranker Model vs Dataset Performance', fontsize=14, fontweight='bold')
plt.xlabel('Reranker Model', fontsize=12)
plt.ylabel('Dataset', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reranker_heatmap.png', dpi=300, bbox_inches='tight')
print(f"✅ Heatmap saved to: {OUTPUT_DIR / 'reranker_heatmap.png'}")
plt.show()

## 📊 Per-Dataset Best Reranker

In [ ]:
# Find best reranker per dataset
best_per_dataset = []

print("\n" + "="*80)
print("🎯 BEST RERANKER PER DATASET")
print("="*80)
print(f"\n{'Dataset':<15} {'Best Reranker':<25} {'NDCG@10':<12}")
print("-"*80)

for dataset in DATASETS:
    dataset_results = results_df[results_df['dataset'] == dataset]
    
    if len(dataset_results) > 0:
        best_row = dataset_results.loc[dataset_results['ndcg_10'].idxmax()]
        print(f"{dataset:<15} {best_row['display_name']:<25} {best_row['ndcg_10']:<12.4f}")
        
        best_per_dataset.append({
            'dataset': dataset,
            'best_reranker': best_row['display_name'],
            'ndcg_10': best_row['ndcg_10']
        })

print("-"*80)

# Save
best_per_dataset_df = pd.DataFrame(best_per_dataset)
best_per_dataset_df.to_csv(OUTPUT_DIR / 'best_reranker_per_dataset.csv', index=False)
print(f"\n✅ Best reranker per dataset saved to: {OUTPUT_DIR / 'best_reranker_per_dataset.csv'}")

## 📝 Summary Report

In [ ]:
# Generate summary report
print("\n" + "="*80)
print("📝 RERANKER EVALUATION SUMMARY REPORT")
print("="*80)

print("\n🔍 Key Findings:")
print("-"*80)

print(f"\n1. Overall Best Reranker:")
print(f"   Model: {best_model['display_name']}")
print(f"   Average NDCG@10: {best_model['ndcg_10']:.4f}")
print(f"   Average Rerank Time: {best_model['avg_rerank_time']:.3f}s")

print(f"\n2. Reranker Rankings (by NDCG@10):")
for i, row in enumerate(reranker_performance.itertuples(), 1):
    print(f"   [{i}] {row.display_name}: {row.ndcg_10:.4f}")

print(f"\n3. Best Reranker per Dataset:")
for item in best_per_dataset:
    print(f"   - {item['dataset']:<15}: {item['best_reranker']} ({item['ndcg_10']:.4f})")

# Count which model is best most often
from collections import Counter
model_counts = Counter([item['best_reranker'] for item in best_per_dataset])
print(f"\n4. Most Versatile Reranker (best on most datasets):")
for model, count in model_counts.most_common():
    print(f"   - {model}: {count}/{len(DATASETS)} datasets")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE!")
print("="*80)

# Save report
report_path = OUTPUT_DIR / 'reranker_evaluation_report.txt'
with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("RERANKER MODEL EVALUATION REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Overall Best Reranker: {best_model['display_name']}\n")
    f.write(f"  Average NDCG@10: {best_model['ndcg_10']:.4f}\n")
    f.write(f"  Average Rerank Time: {best_model['avg_rerank_time']:.3f}s\n\n")
    f.write(f"Reranker Rankings:\n")
    for i, row in enumerate(reranker_performance.itertuples(), 1):
        f.write(f"  [{i}] {row.display_name}: {row.ndcg_10:.4f}\n")
    f.write(f"\nBest Reranker per Dataset:\n")
    for item in best_per_dataset:
        f.write(f"  {item['dataset']}: {item['best_reranker']} ({item['ndcg_10']:.4f})\n")

print(f"\n📄 Report saved to: {report_path}")